In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd

from sklearn.ensemble         import RandomForestClassifier, StackingClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.model_selection  import StratifiedKFold
from sklearn.metrics          import roc_auc_score, recall_score, f1_score
from xgboost                  import XGBClassifier
from lightgbm                 import LGBMClassifier

from utils.preprocessing        import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils           import get_model_train_eval
from utils.feature_engineering  import drop_highly_correlated_features


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def santander_base_job2(columns=['ID'], zcr=0.99, savedDrop=False, 
                       isScaled=False, isSplit=False, 
                       apply_log=False, log_threshold=3.0):  # ✅ 추가
    '''
    SantaderBank CS 분석을 위한 기본 작업
    
    Args:
        columns : list, 1차 삭제할 컬럼
        zcr : float, zero count rate 삭제 기준값
        savedDrop : bool, zcr로 삭제한 컬럼 저장 여부 
        isScaled : bool, scale 여부
        isSplit : bool, train/val split 여부
        apply_log : bool, log1p 변환 적용 여부 (NEW)
        log_threshold : float, log 변환 적용할 skewness 기준값 (NEW)
    
    Returns:   
        isSplit=True  => X_train, X_val, y_train, y_val 
        isSplit=False => X_reduced, y_labels, X_test_reduced
    
    Example:
        # 기본 사용
        X_reduced, y_labels, X_test_reduced = santander_base_job2()
        
        # log 변환 + scaling + split
        X_train, X_val, y_train, y_val = santander_base_job2(
            isScaled=True, 
            isSplit=True, 
            apply_log=True,
            log_threshold=3.0
        )
    '''
    # 데이터 로딩 및 기본 전처리
    train, test = load_data()
    X_features, y_labels = split_features_target(train)
    X_test = test.drop(columns=columns, axis=1)

    # zero_count_rate 제거
    X_features, X_test = remove_zero_columns2(X_features, X_test, zcr)  
    print(f"Zero columns 제거 후: {X_features.shape}")

    # var3 처리
    X_features['var3'] = X_features['var3'].replace(-999999, 2)
    X_test['var3'] = X_test['var3'].replace(-999999, 2)
    
    # 상관계수 높은 feature들 삭제하기
    X_reduced, to_drop = drop_highly_correlated_features(X_features)
    X_test_reduced = X_test.drop(to_drop, axis=1)  
    print(f"상관계수 높은 컬럼 제거 후: {X_reduced.shape}, {X_test_reduced.shape}")
    print(f"삭제된 컬럼 개수: {len(to_drop)}")
    
    if savedDrop:
        series = pd.Series(sorted(to_drop), name="Dropped_Columns")
        series.to_csv("../doc/DroppedColumns.csv", index=False)  
        print(f"✓ 삭제된 컬럼 목록 저장 완료")
    
    # ========================================
    # ✅ NEW: Log 변환 (스케일링 전에!)
    # ========================================
    if apply_log:
        print(f"\n{'='*60}")
        print(f"Log1p 변환 적용 (skewness > {log_threshold})")
        print(f"{'='*60}")
        
        # Train 데이터 분석
        skewness = X_reduced.skew().sort_values(ascending=False)
        high_skew_cols = skewness[abs(skewness) > log_threshold].index.tolist()
        
        # 음수 값이 있는 컬럼 제외
        valid_log_cols = []
        for col in high_skew_cols:
            if (X_reduced[col] >= 0).all():  # 모든 값이 0 이상
                valid_log_cols.append(col)
            else:
                print(f"  ⚠️  {col}: 음수 값 존재 -> log 변환 제외")
        
        if len(valid_log_cols) > 0:
            print(f"\nLog 변환 적용할 컬럼 ({len(valid_log_cols)}개):")
            for i, col in enumerate(valid_log_cols[:10], 1):
                print(f"  {i:2d}. {col:20s} (skewness: {skewness[col]:>7.2f})")
            if len(valid_log_cols) > 10:
                print(f"       ... 외 {len(valid_log_cols)-10}개")
            
            # Train에 log 변환 적용
            X_reduced[valid_log_cols] = np.log1p(X_reduced[valid_log_cols])
            
            # Test에도 동일하게 적용
            X_test_reduced[valid_log_cols] = np.log1p(X_test_reduced[valid_log_cols])
            
            print(f"\n✓ Log1p 변환 완료: {len(valid_log_cols)}개 컬럼")
            
            # 변환 후 왜도 확인
            new_skewness = X_reduced[valid_log_cols].skew()
            print(f"\n변환 후 평균 왜도: {abs(skewness[valid_log_cols]).mean():.2f} -> {abs(new_skewness).mean():.2f}")
        else:
            print("✓ Log 변환 적용 가능한 컬럼이 없습니다.")
        
        print(f"{'='*60}\n")
    
    # ========================================
    # 스케일링 및 Split
    # ========================================
    if isSplit:           
        if isScaled:
            # 스케일링 (log 변환 후)
            X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)        
            # 학습/검증 데이터 분리
            X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
            print(f"✓ 스케일링 + Split 완료")
        else:
            # 학습/검증 데이터 분리
            X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)
            print(f"✓ Split 완료")
        
        return X_train, X_val, y_train, y_val 
    else:
        return X_reduced, y_labels, X_test_reduced

In [3]:
rf_best_param = {
    "random_state": 23,
    "n_estimators": 390,
    "max_depth": 25,
    "class_weight": {0: 1, 1: 2},
    "min_samples_leaf": 1,
    "min_samples_split": 7,
    "n_jobs": -1
}


In [4]:
lr_best_param = {
    "random_state": 23,
    'C': np.float64(0.040779926643605094), 
    'class_weight': None, 
    'max_iter': int(500.0), 
    'penalty': 'l2', 
    'solver': 'lbfgs',
    "n_jobs": -1
}
    # 'C': np.float64(0.040779926643605094), 
    # 'class_weight': np.int64(1), 
    # 'max_iter': np.float64(500.0), 
    # 'penalty': np.int64(0), 
    # 'solver': np.int64(0)}

In [5]:
xgb_best_param = {
    'random_state' : 23,
    'colsample_bytree': round(0.7009895498566958,2), 
    'gamma': round(8.04439822362272,2), 
    'learning_rate': round(0.01019097677223362,2), 
    'max_depth': int(5.0), 
    'min_child_weight': int(8.0), 
    'n_estimators': int(450.0), 
    # 'reg_alpha': round(0.00021647425906756723), 거의 0 
    'reg_lambda': round(2.3459719580898137,2), 
    'scale_pos_weight': int(5.0), 
    'subsample': round(0.6896286494929481,2),
    'use_label_encoder':False,
    'eval_metric':'logloss',
    'n_jobs' : -1
}
    # 'colsample_bytree': np.float64(0.7009895498566958), 
    # 'gamma': np.float64(8.04439822362272), 
    # 'learning_rate': np.float64(0.01019097677223362), 
    # 'max_depth': np.float64(5.0), 
    # 'min_child_weight': np.float64(8.0), 
    # 'n_estimators': np.float64(450.0), 
    # 'reg_alpha': np.float64(0.00021647425906756723), 
    # 'reg_lambda': np.float64(2.3459719580898137), 
    # 'scale_pos_weight': np.float64(5.0), 
    # 'subsample': np.float64(0.6896286494929481)}

In [6]:
lgbm_best_param ={
    'random_state' : 23,
    'n_estimators' : 400,
    'num_leaves' : 36,
    'learning_rate' : 0.03,
    'subsample' : 0.9,
    'colsample_bytree' : 0.75,
    'reg_alpha' : 0.6,
    'reg_lambda' : 0.2,
    'class_weight' : {0:1, 1:10},
    'n_jobs' : -1
}

In [ ]:
lgbm_best_param1 = {
    'random_state'      : 0,
    'n_estimators'      : 500,           # 충분히 크게 잡고 early stopping 사용
    'num_leaves'        : 32,           
    'min_child_samples' : 18,            
    'class_weight'      : {0:1, 1:5},    # 양성 비율 더 강조
    'learning_rate'     : 0.05,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'verbose'           :-1
} # 과적합(?)

In [ ]:
# ========================================
# 3. 비교 실험
# ========================================

# Case 1: Log 변환 없이
X_train1, X_val1, y_train1, y_val1 = santander_base_job2(
    isScaled=True, 
    isSplit=True, 
    apply_log=False
)
model_name1 = 'LightGBM(99per95corrBestHO)_noLog_20251125'


lgbm1 = LGBMClassifier(**lgbm_best_param)
# 함수 이용
get_model_train_eval(lgbm1, model_name1, X_train1, X_val1, y_train1, y_val1)
# rf1.fit(X_train1, y_train1)
# pred_proba1 = rf1.predict_proba(X_val1)[:, 1]
# auc1 = roc_auc_score(y_val1, pred_proba1)

# AUC: 0.8429, 정확도: 0.8899, 정밀도: 0.1916, 재현율: 0.5532, F1: 0.2846

# AUC: 0.8356, 정확도: 0.9281, 정밀도: 0.2323, 재현율: 0.3538, F1: 0.2804


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Case 2: Log 변환 적용
X_train2, X_val2, y_train2, y_val2 = santander_base_job2(
    isScaled=True, 
    isSplit=True, 
    apply_log=True,
    log_threshold=3.0
)
model_name2 = 'LightGBM(99per95corrBestHO)_Log1p_20251125'
lgbm1 = LGBMClassifier(**lgbm_best_param)
get_model_train_eval(lgbm1, model_name2, X_train2, X_val2, y_train2, y_val2)



Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8364, 정확도: 0.9291, 정밀도: 0.2419, 재현율: 0.3704, F1: 0.2927
오차행렬:
[[13903   699]
 [  379   223]]
실행 시간: 2.0111441612243652


In [7]:
def analyze_skewness(df, threshold=3.0):
    """
    데이터프레임의 왜도를 분석하고 log 변환 필요 여부 판단
    
    Args:
        df : pd.DataFrame
        threshold : float, 왜도 기준값
    """
    skewness = df.skew().sort_values(ascending=False)
    
    print(f"{'='*80}")
    print(f"왜도(Skewness) 분석")
    print(f"{'='*80}")
    print(f"전체 컬럼 수: {len(df.columns)}")
    print(f"왜도 > {threshold}: {len(skewness[abs(skewness) > threshold])}개")
    print(f"왜도 > 5: {len(skewness[abs(skewness) > 5])}개")
    print(f"왜도 > 10: {len(skewness[abs(skewness) > 10])}개")
    
    print(f"\n상위 20개 컬럼:")
    print(f"{'-'*80}")
    print(f"{'순위':>4} {'컬럼명':20} {'왜도':>10} {'최소값':>12} {'최대값':>12} {'Log변환':>10}")
    print(f"{'-'*80}")
    
    for i, (col, skew_val) in enumerate(skewness.head(20).items(), 1):
        min_val = df[col].min()
        max_val = df[col].max()
        log_ok = "가능" if min_val >= 0 else "불가(음수)"
        
        print(f"{i:4d} {col:20s} {skew_val:10.2f} {min_val:12.2f} {max_val:12.2f} {log_ok:>10}")
    
    print(f"{'='*80}\n")
    
    # 음수 값 분석
    negative_cols = [col for col in df.columns if (df[col] < 0).any()]
    print(f"음수 값이 있는 컬럼: {len(negative_cols)}개")
    if len(negative_cols) > 0:
        print(f"  예시: {negative_cols[:5]}")


# 사용
# X_features, y_labels = split_features_target(train)
# analyze_skewness(X_features, threshold=3.0)

In [8]:
# 사용
train, test = load_data()
X_features, y_labels = split_features_target(train)
analyze_skewness(X_features, threshold=3.0)

왜도(Skewness) 분석
전체 컬럼 수: 369
왜도 > 3.0: 315개
왜도 > 5: 293개
왜도 > 10: 242개

상위 20개 컬럼:
--------------------------------------------------------------------------------
  순위 컬럼명                          왜도          최소값          최대값      Log변환
--------------------------------------------------------------------------------
   1 num_reemb_var33_ult1     275.72         0.00         3.00         가능
   2 num_reemb_var17_hace3     275.72         0.00         3.00         가능
   3 imp_trasp_var33_out_ult1     275.72         0.00      3000.00         가능
   4 num_trasp_var33_out_ult1     275.72         0.00         3.00         가능
   5 imp_reemb_var33_ult1     275.72         0.00      1200.00         가능
   6 delta_num_trasp_var33_out_1y3     275.72         0.00 9999999999.00         가능
   7 delta_imp_trasp_var33_out_1y3     275.72         0.00 9999999999.00         가능
   8 saldo_medio_var29_hace3     275.72         0.00       145.20         가능
   9 delta_num_aport_var33_1y3     275.72        -1.00 99

In [9]:
X_reduced, y_labels, X_test_reduced = santander_base_job2(
    isScaled=True, 
    isSplit=False, 
    apply_log=True,
    log_threshold=3.0  # skewness > 3인 컬럼만 변환
)


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [10]:
# 모델 앙상블

# 1. Base models 정의
# 최적 하이퍼파라미터: 
# {'max_depth': np.float64(25.0), 
# 'min_samples_leaf': np.float64(1.0), 
# 'min_samples_split': np.float64(7.0), 
# 'n_estimators': np.float64(390.0)}
rf_clf = RandomForestClassifier(**rf_best_param)

# 최적 하이퍼파라미터: 
# 'n_estimators': np.float64(320.0), 
# 'colsample_bytree': np.float64(0.8828333771389649), 
# 'gamma': np.float64(0.058408742978283044), 
# 'learning_rate': np.float64(0.12721361071578832), 
# 'max_depth': np.float64(6.0), 
# 'min_child_weight': np.float64(2.0), 
# 'subsample': np.float64(0.845580758889997)}
xgb_clf = XGBClassifier(
    # random_state      = 23,    
    # n_estimators      = 320,
    # colsample_bytree  = 0.88,
    # gamma             = 0.058, 
    # learning_rate     = 0.13,
    # max_depth         = 6,
    # scale_pos_weight  = 10,
    # min_child_weight  = 2, 
    # subsample         = 0.85,    
    # eval_metric       ='auc',
    # use_label_encoder = False,
    # n_jobs            = -1,
    **xgb_best_param
)

# 최적 하이퍼파라미터: {
# 'n_estimators': np.float64(660.0), 
# 'colsample_bytree': np.float64(0.7342406115797382), 
# 'learning_rate': np.float64(0.02751826185901235), 
# 'num_leaves': np.float64(42.0), 
# 'reg_alpha': np.float64(0.6127977982911577), 
# 'reg_lambda': np.float64(0.1262561992869149), 
# 'subsample': np.float64(0.973776538266153)}
lgbm_clf = LGBMClassifier(**lgbm_best_param)

# 2. Meta model 정의 - 윤지훈님 best param.
meta_model = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    C            = 0.029,
    penalty      = 'l2',
    solver       = 'lbfgs',
    class_weight ="balanced"
    # **lr_best_param
)

# 3. StackingClassifier 구성
stacking_model = StackingClassifier(
    estimators      = [('rf', rf_clf), ('xgb', xgb_clf), ('lgbm', lgbm_clf)],
    final_estimator = meta_model,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs          = -1
)

# 4. 학습 (예시: 전처리된 데이터 X_reduced, y_labels 사용)
stacking_model.fit(X_reduced, y_labels)


,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegre...ndom_state=23)
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,390
,criterion,'gini'
,max_depth,25
,min_samples_split,7


In [11]:
# 5. 메타 모델 계수 출력
coef = stacking_model.final_estimator_.coef_[0]
base_models = ['RandomForest', 'XGBoost', 'LightGBM']
for name, weight in zip(base_models, coef):
    print(f"{name} 기여도(계수): {weight:.4f}")

RandomForest 기여도(계수): 0.4430
XGBoost 기여도(계수): 3.5038
LightGBM 기여도(계수): 2.3344


In [12]:
# 2. 스택킹 모델 성능 평가 (전처리된 데이터 X_reduced, y_labels 사용)
from sklearn.metrics import roc_auc_score, f1_score, recall_score

y_pred_proba = stacking_model.predict_proba(X_reduced)[:, 1]
y_pred = stacking_model.predict(X_reduced)

auc_score = roc_auc_score(y_labels, y_pred_proba)
f1 = f1_score(y_labels, y_pred)
recall = recall_score(y_labels, y_pred)

print(f"\nStacking 모델 ROC-AUC: {auc_score:.4f}")
print(f"Stacking 모델 F1-Score: {f1:.4f}")
print(f"Stacking 모델 Recall:   {recall:.4f}")


Stacking 모델 ROC-AUC: 0.9080
Stacking 모델 F1-Score: 0.2819
Stacking 모델 Recall:   0.8371


In [13]:
from utils.model_utils import save_model

# save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG')
# save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-LR')
save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG_LR')

✓ 모델 저장 완료: ../models\StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG_LR.pkl
  파일 크기: 96.85 MB


'../models\\StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG_LR.pkl'

In [ ]:
# save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG')
# RandomForest 기여도(계수): 0.4430
# XGBoost 기여도(계수): 3.5038
# LightGBM 기여도(계수): 2.3344

# Stacking 모델 ROC-AUC: 0.9080
# Stacking 모델 F1-Score: 0.2819
# Stacking 모델 Recall:   0.8371


# save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-LR') ★★★
# RandomForest 기여도(계수): 1.2551
# XGBoost 기여도(계수): 0.5713
# LightGBM 기여도(계수): 3.2335

# Stacking 모델 ROC-AUC: 0.9416
# Stacking 모델 F1-Score: 0.0073
# Stacking 모델 Recall:   0.0037


# save_model(stacking_model, 'StackingModel_log1p_RF+LGBM+XGB+LR_20251125-XG_LR')
# RandomForest 기여도(계수): 0.8663
# XGBoost 기여도(계수): 2.2970
# LightGBM 기여도(계수): 2.3563

# Stacking 모델 ROC-AUC: 0.9189
# Stacking 모델 F1-Score: 0.0073
# Stacking 모델 Recall:   0.0037